# Flight Delay Data Quality Profile

Notebook này dùng **PySpark** để kiểm tra chất lượng của dữ liệu
**typed raw Parquet** trước khi chuyển sang feature engineering và modeling.

### Mục tiêu
- xác minh schema và quy mô dữ liệu;
- đo missing / blank;
- kiểm tra uniqueness, validity và cardinality;
- kiểm tra referential integrity với lookup tables;
- xác minh target, eligibility và leakage;
- tổng hợp các vấn đề cần bàn giao cho module tiếp theo.

### Không thực hiện trong notebook này
- imputation hoặc drop missing;
- xử lý outlier;
- airport-code mapping;
- feature engineering;
- train/test split;
- model training.

> **Lưu ý:** Typed Parquet vẫn là **typed raw data**, chưa phải cleaned hoặc model-ready dataset.

## 1. Runtime và Spark configuration

Trước khi profiling dữ liệu, cần xác minh notebook đang chạy đúng
Python/PySpark environment và có thể truy cập typed Parquet.

In [1]:
import os
import sys
from pathlib import Path

import pyspark
from pyspark.sql import SparkSession, functions as F

In [2]:
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("FlightDataQualityProfile")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "12")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

26/09/09 10:18:38 WARN Utils: Your hostname, DESKTOP-VVL6IQE resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/09 10:18:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/09 10:18:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print("Runtime")
print("-" * 40)
print(f"Python  : {sys.version.split()[0]}")
print(f"PySpark : {pyspark.__version__}")
print(f"Spark   : {spark.version}")
print(f"Python executable: {sys.executable}")

Runtime
----------------------------------------
Python  : 3.11.16
PySpark : 3.5.9
Spark   : 3.5.9
Python executable: /home/admins/miniforge3/envs/bigdata-spark/bin/python


### Data source

Notebook đọc trực tiếp từ typed Parquet đã được tạo bởi
`scripts/prepare_data.py`.


In [4]:
from pathlib import Path
import os

# 1. Ưu tiên environment variable nếu có
env_path = os.environ.get("FLIGHT_TYPED_PARQUET_DIR")

if env_path:
    data_root = Path(env_path)

else:
    # 2. Nếu không có env var, tìm repo root trong cwd hoặc vài thư mục cha gần nhất
    current = Path.cwd()

    repo_root = None

    for candidate in [current, *current.parents]:
        if (
            (candidate / "requirements.txt").is_file()
            and (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            repo_root = candidate
            break

    if repo_root is None:
        raise RuntimeError("Cannot locate repository root.")

    data_root = repo_root / "data" / "typed-parquet"


paths = {
    "flights": data_root / "flights",
    "airports": data_root / "airports",
    "airlines": data_root / "airlines",
}

missing = [
    name
    for name, path in paths.items()
    if not path.is_dir()
]

if missing:
    raise FileNotFoundError(
        f"Missing typed Parquet datasets: {missing}"
    )

print("Data source check")
print("-" * 40)
print("Status   : PASS")
print("Datasets : flights, airports, airlines")

Data source check
----------------------------------------
Status   : PASS
Datasets : flights, airports, airlines


## 2. Load typed Parquet và kiểm kê dữ liệu

Phần này đọc ba bảng typed Parquet và xác minh quy mô cơ bản của dữ liệu:

- `flights`: bảng chính chứa thông tin chuyến bay;
- `airports`: bảng lookup sân bay;
- `airlines`: bảng lookup hãng bay.

Mục tiêu là xác nhận dữ liệu đọc được đúng và row count khớp với bước `prepare_data.py`.

In [5]:
flights = spark.read.parquet(str(paths["flights"]))
airports = spark.read.parquet(str(paths["airports"]))
airlines = spark.read.parquet(str(paths["airlines"]))

print("Typed Parquet loaded successfully.")

Typed Parquet loaded successfully.


In [6]:
inventory_data = [
    ("flights", flights.count(), len(flights.columns)),
    ("airports", airports.count(), len(airports.columns)),
    ("airlines", airlines.count(), len(airlines.columns)),
]

inventory_df = spark.createDataFrame(
    inventory_data,
    ["dataset", "rows", "columns"]
)

inventory_df.show(truncate=False)

+--------+-------+-------+
|dataset |rows   |columns|
+--------+-------+-------+
|flights |5819079|31     |
|airports|322    |7      |
|airlines|14     |2      |
+--------+-------+-------+



### Schema và dữ liệu mẫu

Schema giúp xác minh các cột numeric đã được cast đúng kiểu trong typed Parquet.

Chỉ hiển thị một số dòng mẫu để kiểm tra cấu trúc dữ liệu, không dùng sample này để đánh giá model.

In [7]:
flights.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: string (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: string (nullable = true)
 |-- DEPARTURE_TIME: string (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: string (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: string (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: string (nullable = true)
 |-- ARRIVAL_TIME: string (nullable = true)
 |-- ARRIVAL_DELAY: integer (nullable = true)
 |-- DIVERTED: integer (nullable

In [8]:
print("Flights sample")
flights.show(5, truncate=False)

print("Airports sample")
airports.show(5, truncate=False)

print("Airlines sample")
airlines.show(5, truncate=False)

Flights sample


26/09/09 10:21:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+-----+
|YEAR|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|MONTH|
+----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-------

### Conclusion

- Cả ba typed Parquet datasets đều được đọc thành công.
- Row count khớp với kết quả validation trước đó:
  - flights: 5,819,079
  - airports: 322
  - airlines: 14
- Schema đã được typed, nên dữ liệu sẵn sàng cho bước profiling chất lượng dữ liệu tiếp theo.

## 3. Completeness: Null và Blank

Phần này đo mức độ đầy đủ của dữ liệu theo hai dạng:

- **Null**: giá trị bị thiếu ở mức schema;
- **Blank**: chuỗi rỗng hoặc chỉ chứa khoảng trắng.

Hai khái niệm được kiểm tra riêng để tránh gộp nhầm missing dạng null với
missing dạng text.


In [9]:
from pyspark.sql import types as T

def completeness_profile(df):
    total_rows = df.count()

    null_exprs = [
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]

    null_counts = df.agg(*null_exprs).first().asDict()

    string_cols = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, T.StringType)
    ]

    if string_cols:
        blank_exprs = [
            F.sum(
                (
                    F.col(c).isNotNull()
                    & (F.trim(F.col(c)) == "")
                ).cast("int")
            ).alias(c)
            for c in string_cols
        ]

        blank_counts = df.agg(*blank_exprs).first().asDict()
    else:
        blank_counts = {}

    rows = []

    for c in df.columns:
        null_count = int(null_counts[c] or 0)
        blank_count = int(blank_counts.get(c, 0) or 0)

        rows.append(
            (
                c,
                null_count,
                round(null_count / total_rows * 100, 4),
                blank_count,
                round(blank_count / total_rows * 100, 4),
            )
        )

    return spark.createDataFrame(
        rows,
        [
            "column",
            "null_count",
            "null_rate_pct",
            "blank_count",
            "blank_rate_pct",
        ],
    )

In [10]:
flights_completeness = completeness_profile(flights)

flights_completeness.orderBy(
    F.desc("null_rate_pct")
).show(31, truncate=False)

+-------------------+----------+-------------+-----------+--------------+
|column             |null_count|null_rate_pct|blank_count|blank_rate_pct|
+-------------------+----------+-------------+-----------+--------------+
|CANCELLATION_REASON|5729195   |98.4554      |0          |0.0           |
|AIR_SYSTEM_DELAY   |4755640   |81.725       |0          |0.0           |
|SECURITY_DELAY     |4755640   |81.725       |0          |0.0           |
|AIRLINE_DELAY      |4755640   |81.725       |0          |0.0           |
|LATE_AIRCRAFT_DELAY|4755640   |81.725       |0          |0.0           |
|WEATHER_DELAY      |4755640   |81.725       |0          |0.0           |
|ELAPSED_TIME       |105071    |1.8056       |0          |0.0           |
|AIR_TIME           |105071    |1.8056       |0          |0.0           |
|ARRIVAL_DELAY      |105071    |1.8056       |0          |0.0           |
|WHEELS_ON          |92513     |1.5898       |0          |0.0           |
|TAXI_IN            |92513     |1.5898

In [11]:
airports_completeness = completeness_profile(airports)

airports_completeness.orderBy(
    F.desc("null_rate_pct")
).show(truncate=False)

+---------+----------+-------------+-----------+--------------+
|column   |null_count|null_rate_pct|blank_count|blank_rate_pct|
+---------+----------+-------------+-----------+--------------+
|LATITUDE |3         |0.9317       |0          |0.0           |
|LONGITUDE|3         |0.9317       |0          |0.0           |
|IATA_CODE|0         |0.0          |0          |0.0           |
|STATE    |0         |0.0          |0          |0.0           |
|AIRPORT  |0         |0.0          |0          |0.0           |
|COUNTRY  |0         |0.0          |0          |0.0           |
|CITY     |0         |0.0          |0          |0.0           |
+---------+----------+-------------+-----------+--------------+



In [12]:
airlines_completeness = completeness_profile(airlines)

airlines_completeness.show(truncate=False)

+---------+----------+-------------+-----------+--------------+
|column   |null_count|null_rate_pct|blank_count|blank_rate_pct|
+---------+----------+-------------+-----------+--------------+
|IATA_CODE|0         |0.0          |0          |0.0           |
|AIRLINE  |0         |0.0          |0          |0.0           |
+---------+----------+-------------+-----------+--------------+



### Conclusion

- `ARRIVAL_DELAY` thiếu 105,071 dòng, khoảng 1.81% tổng số chuyến bay.
- Các cột nguyên nhân delay thiếu khoảng 81.7%; đây là missing có ý nghĩa nghiệp vụ và đồng thời là post-outcome information.
- `LATITUDE` và `LONGITUDE` của bảng airports cùng thiếu ở 3 dòng.
- Không phát hiện blank/whitespace đáng kể trong dữ liệu typed raw.

Các missing này **chưa được xử lý trong notebook**; phần này chỉ ghi nhận để bàn giao cho module phù hợp.

## 4. Missing Pattern

Sau khi biết cột nào có missing, phần này kiểm tra xem các missing đó có xuất hiện theo một quy luật rõ ràng hay không.

Hai pattern chính cần xác minh:

1. `ARRIVAL_DELAY` bị null có liên quan đến các chuyến `CANCELLED` hoặc `DIVERTED` hay không.
2. Các delay-cause columns chỉ xuất hiện khi chuyến bay đã bị trễ đáng kể hay không.

In [13]:
arrival_missing_pattern = flights.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.col("ARRIVAL_DELAY").isNull().cast("int")
    ).alias("arrival_delay_null"),

    F.sum(
        (F.col("CANCELLED") == 1).cast("int")
    ).alias("cancelled"),

    F.sum(
        (F.col("DIVERTED") == 1).cast("int")
    ).alias("diverted"),

    F.sum(
        (
            F.col("ARRIVAL_DELAY").isNull()
            & (F.col("CANCELLED") == 0)
            & (F.col("DIVERTED") == 0)
        ).cast("int")
    ).alias("arrival_null_but_not_cancelled_or_diverted"),
)

arrival_missing_pattern.show(truncate=False)

+----------+------------------+---------+--------+------------------------------------------+
|total_rows|arrival_delay_null|cancelled|diverted|arrival_null_but_not_cancelled_or_diverted|
+----------+------------------+---------+--------+------------------------------------------+
|5819079   |105071            |89884    |15187   |0                                         |
+----------+------------------+---------+--------+------------------------------------------+



### Conclusion: `ARRIVAL_DELAY`

- Có 105,071 dòng thiếu `ARRIVAL_DELAY`.
- Toàn bộ các dòng này thuộc chuyến `CANCELLED` hoặc `DIVERTED`.
- Không có trường hợp `ARRIVAL_DELAY` null nằm ngoài hai nhóm trên.

Vì vậy, đây không phải missing ngẫu nhiên. Nó gắn với trạng thái nghiệp vụ của chuyến bay và sẽ ảnh hưởng trực tiếp đến việc xác định **eligible rows** cho hai bài toán model.

### Delay-cause columns

Các delay-cause columns mô tả nguyên nhân của delay và có tỷ lệ missing rất cao.

Phần này kiểm tra xem các giá trị non-null có xuất hiện chủ yếu khi
`ARRIVAL_DELAY >= 15` hay không.

In [14]:
delay_cause_cols = [
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
]

delay_pattern_exprs = []

for c in delay_cause_cols:
    delay_pattern_exprs.extend([
        F.sum(
            F.col(c).isNotNull().cast("int")
        ).alias(f"{c}_nonnull"),

        F.sum(
            (
                F.col(c).isNotNull()
                & (F.col("ARRIVAL_DELAY") < 15)
            ).cast("int")
        ).alias(f"{c}_nonnull_when_arrival_delay_lt15"),
    ])

delay_pattern = flights.agg(*delay_pattern_exprs).first().asDict()

In [15]:
delay_pattern_rows = []

for c in delay_cause_cols:
    nonnull = delay_pattern[f"{c}_nonnull"]
    below_15 = delay_pattern[f"{c}_nonnull_when_arrival_delay_lt15"]

    delay_pattern_rows.append(
        (c, nonnull, below_15)
    )

delay_pattern_df = spark.createDataFrame(
    delay_pattern_rows,
    [
        "column",
        "nonnull_count",
        "nonnull_when_arrival_delay_lt_15",
    ],
)

delay_pattern_df.show(truncate=False)

+-------------------+-------------+--------------------------------+
|column             |nonnull_count|nonnull_when_arrival_delay_lt_15|
+-------------------+-------------+--------------------------------+
|AIR_SYSTEM_DELAY   |1063439      |0                               |
|SECURITY_DELAY     |1063439      |0                               |
|AIRLINE_DELAY      |1063439      |0                               |
|LATE_AIRCRAFT_DELAY|1063439      |0                               |
|WEATHER_DELAY      |1063439      |0                               |
+-------------------+-------------+--------------------------------+



### Conclusion: Delay-cause columns

- Các delay-cause columns thiếu khoảng 81.7% dữ liệu.
- Các giá trị non-null chỉ xuất hiện khi `ARRIVAL_DELAY >= 15`.
- Missing ở đây có ý nghĩa nghiệp vụ: các nguyên nhân delay chỉ được ghi khi chuyến bay đã đủ trễ để cần phân rã nguyên nhân.
- Các cột này đồng thời là **post-outcome leakage**, vì chỉ biết sau khi delay đã xảy ra.

Do đó không nên tự động fill null bằng 0 trong bước data preparation.

## 5. Uniqueness và Duplicate

Phần này kiểm tra tính duy nhất của các khóa lookup và trạng thái duplicate.

- `airports.IATA_CODE` phải là khóa duy nhất của bảng sân bay.
- `airlines.IATA_CODE` phải là khóa duy nhất của bảng hãng bay.
- Với `flights`, exact full-row duplicate đã được kiểm tra ở bước profiling raw trước đó; notebook này không chạy lại nếu không cần thiết để tránh lặp một phép kiểm tra nặng.

In [16]:
def lookup_uniqueness(df, key_col):
    result = df.agg(
        F.count("*").alias("rows"),
        F.countDistinct(key_col).alias("distinct_keys"),
        F.sum(
            (
                F.col(key_col).isNull()
                | (F.trim(F.col(key_col)) == "")
            ).cast("int")
        ).alias("missing_keys"),
    ).first()

    duplicate_keys = result["rows"] - result["distinct_keys"]

    return (
        result["rows"],
        result["distinct_keys"],
        result["missing_keys"],
        duplicate_keys,
    )

In [17]:
airports_stats = lookup_uniqueness(airports, "IATA_CODE")
airlines_stats = lookup_uniqueness(airlines, "IATA_CODE")

uniqueness_rows = [
    ("airports", *airports_stats),
    ("airlines", *airlines_stats),
]

uniqueness_df = spark.createDataFrame(
    uniqueness_rows,
    [
        "dataset",
        "rows",
        "distinct_keys",
        "missing_keys",
        "duplicate_keys",
    ]
)

uniqueness_df.show(truncate=False)

+--------+----+-------------+------------+--------------+
|dataset |rows|distinct_keys|missing_keys|duplicate_keys|
+--------+----+-------------+------------+--------------+
|airports|322 |322          |0           |0             |
|airlines|14  |14           |0           |0             |
+--------+----+-------------+------------+--------------+



### Conclusion

- `airports.IATA_CODE` và `airlines.IATA_CODE` đều không thiếu và không trùng khóa.
- Hai lookup table đáp ứng yêu cầu uniqueness để dùng cho join.
- Exact full-row duplicate của `flights` không được chạy lại trong notebook; trạng thái được ghi rõ là `SKIPPED` vì đã được profile trước trên raw data.

## 6. Validity và Domain Checks

Phần này kiểm tra xem các giá trị có nằm trong domain hợp lệ hay không.

Các rule chính:

- `YEAR = 2015`
- `MONTH ∈ [1, 12]`
- `DAY ∈ [1, 31]`
- `DAY_OF_WEEK ∈ [1, 7]`
- `CANCELLED ∈ {0, 1}`
- `DIVERTED ∈ {0, 1}`
- `LATITUDE ∈ [-90, 90]`
- `LONGITUDE ∈ [-180, 180]`

In [19]:
domain_checks = [
    ("YEAR", ~F.col("YEAR").isin([2015])),
    ("MONTH", ~F.col("MONTH").between(1, 12)),
    ("DAY", ~F.col("DAY").between(1, 31)),
    ("DAY_OF_WEEK", ~F.col("DAY_OF_WEEK").between(1, 7)),
    ("CANCELLED", ~F.col("CANCELLED").isin([0, 1])),
    ("DIVERTED", ~F.col("DIVERTED").isin([0, 1])),
]

domain_results = []

for column, invalid_condition in domain_checks:
    invalid_count = flights.filter(
        F.col(column).isNotNull() & invalid_condition
    ).count()

    domain_results.append(
        (
            column,
            invalid_count,
            "PASS" if invalid_count == 0 else "FAIL",
        )
    )

domain_results_df = spark.createDataFrame(
    domain_results,
    ["column", "invalid_count", "status"]
)

domain_results_df.show(truncate=False)

+-----------+-------------+------+
|column     |invalid_count|status|
+-----------+-------------+------+
|YEAR       |0            |PASS  |
|MONTH      |0            |PASS  |
|DAY        |0            |PASS  |
|DAY_OF_WEEK|0            |PASS  |
|CANCELLED  |0            |PASS  |
|DIVERTED   |0            |PASS  |
+-----------+-------------+------+



In [20]:
coordinate_check = airports.agg(
    F.sum(
        (
            F.col("LATITUDE").isNotNull()
            & ~F.col("LATITUDE").between(-90, 90)
        ).cast("int")
    ).alias("invalid_latitude"),

    F.sum(
        (
            F.col("LONGITUDE").isNotNull()
            & ~F.col("LONGITUDE").between(-180, 180)
        ).cast("int")
    ).alias("invalid_longitude"),

    F.sum(
        (
            F.col("LATITUDE").isNull()
            & F.col("LONGITUDE").isNull()
        ).cast("int")
    ).alias("both_coordinates_null"),

    F.sum(
        (
            F.col("LATITUDE").isNull()
            != F.col("LONGITUDE").isNull()
        ).cast("int")
    ).alias("only_one_coordinate_null"),
)

coordinate_check.show(truncate=False)

+----------------+-----------------+---------------------+------------------------+
|invalid_latitude|invalid_longitude|both_coordinates_null|only_one_coordinate_null|
+----------------+-----------------+---------------------+------------------------+
|0               |0                |3                    |0                       |
+----------------+-----------------+---------------------+------------------------+



### Conclusion

- Không phát hiện giá trị ngoài domain ở các cột thời gian và trạng thái được kiểm tra.
- Không có latitude hoặc longitude nằm ngoài phạm vi địa lý hợp lệ.
- Có 3 sân bay thiếu đồng thời cả `LATITUDE` và `LONGITUDE`.
- Không có trường hợp chỉ thiếu một trong hai tọa độ.

## 7. HHMM Validity

Các cột thời gian hiện vẫn được giữ ở dạng string theo format HHMM.

Phần này kiểm tra:

- giá trị null;
- chuỗi blank;
- giá trị hợp lệ trong khoảng `0000–2359`;
- giá trị đặc biệt `2400`;
- format hoặc range không hợp lệ.

Notebook chỉ kiểm tra validity, không convert sang timestamp.

In [21]:
hhmm_cols = [
    "SCHEDULED_DEPARTURE",
    "DEPARTURE_TIME",
    "WHEELS_OFF",
    "WHEELS_ON",
    "SCHEDULED_ARRIVAL",
    "ARRIVAL_TIME",
]

In [22]:
def hhmm_profile(df, column):
    c = F.col(column)

    valid_format = c.rlike(r"^\d{4}$")

    hour = c.substr(1, 2).cast("int")
    minute = c.substr(3, 2).cast("int")

    valid_regular_time = (
        valid_format
        & hour.between(0, 23)
        & minute.between(0, 59)
    )

    is_2400 = c == "2400"

    result = df.agg(
        F.count("*").alias("rows"),

        F.sum(c.isNull().cast("int")).alias("null_count"),

        F.sum(
            (
                c.isNotNull()
                & (F.trim(c) == "")
            ).cast("int")
        ).alias("blank_count"),

        F.sum(
            valid_regular_time.cast("int")
        ).alias("valid_hhmm_count"),

        F.sum(
            is_2400.cast("int")
        ).alias("value_2400_count"),

        F.sum(
            (
                c.isNotNull()
                & (F.trim(c) != "")
                & ~valid_regular_time
                & ~is_2400
            ).cast("int")
        ).alias("invalid_count"),
    ).first()

    return (
        column,
        result["null_count"],
        result["blank_count"],
        result["valid_hhmm_count"],
        result["value_2400_count"],
        result["invalid_count"],
    )

In [23]:
hhmm_results = [
    hhmm_profile(flights, c)
    for c in hhmm_cols
]

hhmm_df = spark.createDataFrame(
    hhmm_results,
    [
        "column",
        "null_count",
        "blank_count",
        "valid_hhmm_count",
        "value_2400_count",
        "invalid_count",
    ],
)

hhmm_df.show(truncate=False)

+-------------------+----------+-----------+----------------+----------------+-------------+
|column             |null_count|blank_count|valid_hhmm_count|value_2400_count|invalid_count|
+-------------------+----------+-----------+----------------+----------------+-------------+
|SCHEDULED_DEPARTURE|0         |0          |5819079         |0               |0            |
|DEPARTURE_TIME     |86153     |0          |5732413         |513             |0            |
|WHEELS_OFF         |89047     |0          |5729305         |727             |0            |
|WHEELS_ON          |92513     |0          |5724561         |2005            |0            |
|SCHEDULED_ARRIVAL  |0         |0          |5819077         |2               |0            |
|ARRIVAL_TIME       |92513     |0          |5724110         |2456            |0            |
+-------------------+----------+-----------+----------------+----------------+-------------+



### Conclusion

- Không phát hiện giá trị HHMM sai format hoặc ngoài range trong 6 cột thời gian được kiểm tra.
- Giá trị `2400` được ghi nhận riêng và không coi là lỗi theo convention của dataset.
- Các cột actual-time có thể chứa null do trạng thái chuyến bay; đây là vấn đề completeness/business state, không phải lỗi format.

Các cột thời gian vẫn giữ nguyên dạng string trong typed raw.

## 8. Cardinality

Cardinality là số lượng giá trị khác nhau của một cột.

Phần này tập trung vào các cột categorical/identifier quan trọng để nhận diện:

- low-cardinality: phù hợp cho các cách encoding thông thường;
- high-cardinality: cần cân nhắc kỹ trong feature engineering.

Notebook chỉ đo cardinality, chưa thực hiện encoding.

In [24]:
cardinality_cols = [
    "AIRLINE",
    "FLIGHT_NUMBER",
    "TAIL_NUMBER",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "CANCELLATION_REASON",
]

In [25]:
cardinality_exprs = [
    F.countDistinct(c).alias(c)
    for c in cardinality_cols
]

cardinality_stats = (
    flights
    .agg(*cardinality_exprs)
    .first()
    .asDict()
)

In [26]:
cardinality_rows = [
    (column, cardinality_stats[column])
    for column in cardinality_cols
]

cardinality_df = spark.createDataFrame(
    cardinality_rows,
    ["column", "distinct_count"]
)

cardinality_df.orderBy(
    F.desc("distinct_count")
).show(truncate=False)

+-------------------+--------------+
|column             |distinct_count|
+-------------------+--------------+
|FLIGHT_NUMBER      |6952          |
|TAIL_NUMBER        |4897          |
|DESTINATION_AIRPORT|629           |
|ORIGIN_AIRPORT     |628           |
|AIRLINE            |14            |
|CANCELLATION_REASON|4             |
+-------------------+--------------+



### Conclusion

- `AIRLINE` có cardinality thấp, chỉ gồm một số lượng nhỏ hãng bay.
- `ORIGIN_AIRPORT` và `DESTINATION_AIRPORT` có cardinality trung bình.
- `FLIGHT_NUMBER` và đặc biệt `TAIL_NUMBER` có cardinality cao hơn đáng kể.
- `TAIL_NUMBER` vì vậy cần được xem xét thêm trước khi dùng làm feature.
- Cardinality được ghi nhận để bàn giao cho module feature engineering; notebook này chưa thực hiện encoding.

## 9. Referential Integrity và Join Coverage

Phần này kiểm tra mức độ khớp giữa bảng `flights` và hai lookup tables:

- `flights.AIRLINE` → `airlines.IATA_CODE`
- `flights.ORIGIN_AIRPORT` → `airports.IATA_CODE`
- `flights.DESTINATION_AIRPORT` → `airports.IATA_CODE`

Mục tiêu là đo tỷ lệ match/unmatch và xác định nguyên nhân chính của các mã sân bay không match trực tiếp.

Notebook chỉ kiểm tra coverage, chưa thực hiện mapping mã DOT sang IATA.

In [27]:
airline_join = flights.join(
    airlines.select(
        F.col("IATA_CODE").alias("AIRLINE_KEY")
    ),
    flights["AIRLINE"] == F.col("AIRLINE_KEY"),
    "left"
)

airline_coverage = airline_join.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.col("AIRLINE_KEY").isNotNull().cast("int")
    ).alias("matched_rows"),

    F.sum(
        F.col("AIRLINE_KEY").isNull().cast("int")
    ).alias("unmatched_rows"),
).first()

airline_coverage_pct = (
    airline_coverage["matched_rows"]
    / airline_coverage["total_rows"]
    * 100
)

print("Airline lookup coverage")
print("-" * 40)
print(f"Total rows   : {airline_coverage['total_rows']:,}")
print(f"Matched rows : {airline_coverage['matched_rows']:,}")
print(f"Unmatched    : {airline_coverage['unmatched_rows']:,}")
print(f"Coverage     : {airline_coverage_pct:.4f}%")

Airline lookup coverage
----------------------------------------
Total rows   : 5,819,079
Matched rows : 5,819,079
Unmatched    : 0
Coverage     : 100.0000%


In [28]:
def airport_coverage(df, airport_col, airports_df):
    joined = df.join(
        airports_df.select(
            F.col("IATA_CODE").alias("AIRPORT_KEY")
        ),
        df[airport_col] == F.col("AIRPORT_KEY"),
        "left"
    )

    result = joined.agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.col("AIRPORT_KEY").isNotNull().cast("int")
        ).alias("matched_rows"),

        F.sum(
            F.col("AIRPORT_KEY").isNull().cast("int")
        ).alias("unmatched_rows"),
    ).first()

    coverage_pct = (
        result["matched_rows"]
        / result["total_rows"]
        * 100
    )

    return (
        result["total_rows"],
        result["matched_rows"],
        result["unmatched_rows"],
        coverage_pct,
    )

In [29]:
origin_stats = airport_coverage(
    flights,
    "ORIGIN_AIRPORT",
    airports
)

destination_stats = airport_coverage(
    flights,
    "DESTINATION_AIRPORT",
    airports
)

airport_coverage_df = spark.createDataFrame(
    [
        ("ORIGIN_AIRPORT", *origin_stats),
        ("DESTINATION_AIRPORT", *destination_stats),
    ],
    [
        "column",
        "total_rows",
        "matched_rows",
        "unmatched_rows",
        "coverage_pct",
    ]
)

airport_coverage_df.show(truncate=False)

+-------------------+----------+------------+--------------+----------------+
|column             |total_rows|matched_rows|unmatched_rows|coverage_pct    |
+-------------------+----------+------------+--------------+----------------+
|ORIGIN_AIRPORT     |5819079   |5332914     |486165        |91.6453273791265|
|DESTINATION_AIRPORT|5819079   |5332914     |486165        |91.6453273791265|
+-------------------+----------+------------+--------------+----------------+



In [30]:
numeric_airport_by_month = (
    flights
    .groupBy("MONTH")
    .agg(
        F.sum(
            F.col("ORIGIN_AIRPORT")
            .rlike(r"^\d{5}$")
            .cast("int")
        ).alias("numeric_origin_rows"),

        F.sum(
            F.col("DESTINATION_AIRPORT")
            .rlike(r"^\d{5}$")
            .cast("int")
        ).alias("numeric_destination_rows"),
    )
    .orderBy("MONTH")
)

numeric_airport_by_month.show(12, truncate=False)

+-----+-------------------+------------------------+
|MONTH|numeric_origin_rows|numeric_destination_rows|
+-----+-------------------+------------------------+
|1    |0                  |0                       |
|2    |0                  |0                       |
|3    |0                  |0                       |
|4    |0                  |0                       |
|5    |0                  |0                       |
|6    |0                  |0                       |
|7    |0                  |0                       |
|8    |0                  |0                       |
|9    |0                  |0                       |
|10   |486165             |486165                  |
|11   |0                  |0                       |
|12   |0                  |0                       |
+-----+-------------------+------------------------+



In [31]:
origin_unmatched_codes = (
    flights
    .join(
        airports.select(
            F.col("IATA_CODE").alias("AIRPORT_KEY")
        ),
        flights["ORIGIN_AIRPORT"] == F.col("AIRPORT_KEY"),
        "left"
    )
    .filter(F.col("AIRPORT_KEY").isNull())
    .select("ORIGIN_AIRPORT")
    .distinct()
    .count()
)

destination_unmatched_codes = (
    flights
    .join(
        airports.select(
            F.col("IATA_CODE").alias("AIRPORT_KEY")
        ),
        flights["DESTINATION_AIRPORT"] == F.col("AIRPORT_KEY"),
        "left"
    )
    .filter(F.col("AIRPORT_KEY").isNull())
    .select("DESTINATION_AIRPORT")
    .distinct()
    .count()
)

print("Distinct unmatched airport codes")
print("-" * 40)
print(f"Origin      : {origin_unmatched_codes}")
print(f"Destination : {destination_unmatched_codes}")

Distinct unmatched airport codes
----------------------------------------
Origin      : 306
Destination : 307


### Conclusion

- `AIRLINE` match 100% với bảng `airlines`.
- `ORIGIN_AIRPORT` và `DESTINATION_AIRPORT` chỉ match trực tiếp khoảng 91.65% với bảng `airports`.
- Phần lớn unmatch tập trung ở tháng 10, khi dữ liệu xuất hiện mã sân bay numeric/DOT 5 chữ số.
- Bảng `airports` chỉ dùng mã IATA 3 ký tự, nên không thể join trực tiếp các mã DOT này.

Vì vậy, referential integrity của airline lookup là tốt, nhưng airport lookup còn **partial** và cần được module Custom/Feature xử lý bằng mapping DOT → IATA đã được xác minh trước khi tính Haversine.

## 10. Target và Eligibility

Hai bài toán dùng chung target gốc `ARRIVAL_DELAY`.

Một dòng chỉ được xem là **eligible** khi:

- `ARRIVAL_DELAY` không null;
- `CANCELLED = 0`;
- `DIVERTED = 0`.

Phần này chỉ xác định và thống kê eligible rows, chưa tạo train/test split hay model dataset.

In [32]:
eligible_condition = (
    F.col("ARRIVAL_DELAY").isNotNull()
    & (F.col("CANCELLED") == 0)
    & (F.col("DIVERTED") == 0)
)

eligibility_stats = flights.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        eligible_condition.cast("int")
    ).alias("eligible_rows"),

    F.sum(
        (~eligible_condition).cast("int")
    ).alias("excluded_rows"),

    F.sum(
        F.col("ARRIVAL_DELAY").isNull().cast("int")
    ).alias("arrival_delay_null"),

    F.sum(
        (F.col("CANCELLED") == 1).cast("int")
    ).alias("cancelled"),

    F.sum(
        (F.col("DIVERTED") == 1).cast("int")
    ).alias("diverted"),
).first()

eligibility_df = spark.createDataFrame(
    [(
        eligibility_stats["total_rows"],
        eligibility_stats["eligible_rows"],
        eligibility_stats["excluded_rows"],
        eligibility_stats["arrival_delay_null"],
        eligibility_stats["cancelled"],
        eligibility_stats["diverted"],
    )],
    [
        "total_rows",
        "eligible_rows",
        "excluded_rows",
        "arrival_delay_null",
        "cancelled",
        "diverted",
    ]
)

eligibility_df.show(truncate=False)

+----------+-------------+-------------+------------------+---------+--------+
|total_rows|eligible_rows|excluded_rows|arrival_delay_null|cancelled|diverted|
+----------+-------------+-------------+------------------+---------+--------+
|5819079   |5714008      |105071       |105071            |89884    |15187   |
+----------+-------------+-------------+------------------+---------+--------+



### Conclusion

- Có 5,714,008 chuyến bay đủ điều kiện để dùng cho cả hai bài toán.
- Các dòng bị loại chủ yếu là chuyến `CANCELLED` hoặc `DIVERTED`, tương ứng với target `ARRIVAL_DELAY` bị null.


## 11. Classification Label Distribution

Bài toán classification dùng label tạm thời:

- `1` nếu `ARRIVAL_DELAY > 30`;
- `0` nếu `ARRIVAL_DELAY <= 30`.

In [33]:
eligible_flights = flights.filter(eligible_condition)

label_distribution = (
    eligible_flights
    .select(
        F.when(
            F.col("ARRIVAL_DELAY") > 30, 1
        ).otherwise(0).alias("label")
    )
    .groupBy("label")
    .count()
    .orderBy("label")
)

label_distribution.show()

+-----+-------+
|label|  count|
+-----+-------+
|    0|5081311|
|    1| 632697|
+-----+-------+



In [34]:
label_counts = {
    row["label"]: row["count"]
    for row in label_distribution.collect()
}

negative = label_counts.get(0, 0)
positive = label_counts.get(1, 0)
total = negative + positive

positive_rate = positive / total * 100

print("Classification label distribution")
print("-" * 45)
print(f"Negative (<=30 min) : {negative:,}")
print(f"Positive (>30 min)  : {positive:,}")
print(f"Positive rate       : {positive_rate:.4f}%")

Classification label distribution
---------------------------------------------
Negative (<=30 min) : 5,081,311
Positive (>30 min)  : 632,697
Positive rate       : 11.0727%


### Conclusion

- Positive class chỉ chiếm khoảng 11.07%.
- Bài toán classification bị mất cân bằng đáng kể.

## 12. Numeric Descriptive Statistics

Phần này mô tả một số biến numeric quan trọng để hiểu range, trung tâm và độ phân tán của dữ liệu.

Không xử lý outlier trong notebook này.

In [36]:
numeric_cols = [
    "ARRIVAL_DELAY",
    "DEPARTURE_DELAY",
    "SCHEDULED_TIME",
    "DISTANCE",
]

numeric_profile = (
    flights
    .select(numeric_cols)
    .summary(
        "count",
        "min",
        "mean",
        "stddev",
        "max"
    )
)

numeric_profile.show(truncate=False)

+-------+-----------------+-----------------+-----------------+-----------------+
|summary|ARRIVAL_DELAY    |DEPARTURE_DELAY  |SCHEDULED_TIME   |DISTANCE         |
+-------+-----------------+-----------------+-----------------+-----------------+
|count  |5714008          |5732926          |5819073          |5819079          |
|min    |-87              |-82              |18               |21               |
|mean   |4.407057357987598|9.370158275198389|141.6858915500802|822.3564947305235|
|stddev |39.27129709388725|37.08094249678673|75.21058230600217|607.7842873170591|
|max    |1971             |1988             |718              |4983             |
+-------+-----------------+-----------------+-----------------+-----------------+



In [37]:
quantile_rows = []

for column in numeric_cols:
    q25, q50, q75 = flights.approxQuantile(
        column,
        [0.25, 0.50, 0.75],
        0.01
    )

    quantile_rows.append(
        (column, q25, q50, q75)
    )

quantiles_df = spark.createDataFrame(
    quantile_rows,
    ["column", "q25", "median", "q75"]
)

quantiles_df.show(truncate=False)

+---------------+-----+------+------+
|column         |q25  |median|q75   |
+---------------+-----+------+------+
|ARRIVAL_DELAY  |-13.0|-5.0  |8.0   |
|DEPARTURE_DELAY|-5.0 |-1.0  |7.0   |
|SCHEDULED_TIME |87.0 |122.0 |175.0 |
|DISTANCE       |371.0|646.0 |1055.0|
+---------------+-----+------+------+



In [38]:
arrival_stats = eligible_flights.agg(
    F.min("ARRIVAL_DELAY").alias("min"),
    F.max("ARRIVAL_DELAY").alias("max"),
    F.avg("ARRIVAL_DELAY").alias("mean"),
    F.stddev("ARRIVAL_DELAY").alias("stddev"),

    F.sum(
        (F.col("ARRIVAL_DELAY") < 0).cast("int")
    ).alias("negative_rows"),

    F.count("*").alias("rows"),
).first()

negative_pct = (
    arrival_stats["negative_rows"]
    / arrival_stats["rows"]
    * 100
)

print("ARRIVAL_DELAY profile")
print("-" * 40)
print(f"Min              : {arrival_stats['min']}")
print(f"Max              : {arrival_stats['max']}")
print(f"Mean             : {arrival_stats['mean']:.4f}")
print(f"Negative rows    : {arrival_stats['negative_rows']:,}")
print(f"Negative rate    : {negative_pct:.4f}%")

ARRIVAL_DELAY profile
----------------------------------------
Min              : -87
Max              : 1971
Mean             : 4.4071
Negative rows    : 3,500,899
Negative rate    : 61.2687%


### Conclusion

- `ARRIVAL_DELAY` có cả giá trị âm và dương.
- Giá trị âm là hợp lệ về nghiệp vụ: chuyến bay đến sớm hơn lịch.
- Khoảng 61% target có giá trị âm.
- Vì vậy không được clip target âm trong bước chuẩn bị dữ liệu.
- Các model/loss yêu cầu target dương cần được module Model cân nhắc riêng.

## 13. Leakage và Inference Contract

Inference time của project được chọn là:

**trước khi chuyến bay thực sự khởi hành.**

Vì vậy chỉ được sử dụng các thông tin đã biết trước actual departure.
Phần này phân loại các cột theo vai trò để tránh data leakage.

In [39]:
feature_roles = [
    ("YEAR", "Candidate feature", "Pre-departure"),
    ("MONTH", "Candidate feature", "Pre-departure"),
    ("DAY", "Candidate feature", "Pre-departure"),
    ("DAY_OF_WEEK", "Candidate feature", "Pre-departure"),
    ("AIRLINE", "Candidate feature", "Pre-departure"),
    ("FLIGHT_NUMBER", "Candidate feature", "Pre-departure"),
    ("ORIGIN_AIRPORT", "Candidate feature", "Pre-departure"),
    ("DESTINATION_AIRPORT", "Candidate feature", "Pre-departure"),
    ("SCHEDULED_DEPARTURE", "Candidate feature", "Pre-departure"),
    ("SCHEDULED_ARRIVAL", "Candidate feature", "Pre-departure"),
    ("SCHEDULED_TIME", "Candidate feature", "Pre-departure"),
    ("DISTANCE", "Candidate feature", "Pre-departure"),

    ("TAIL_NUMBER", "Requires review", "Availability/high cardinality"),

    ("ARRIVAL_DELAY", "Target", "Outcome"),

    ("CANCELLED", "Eligibility only", "Not feature"),
    ("DIVERTED", "Eligibility only", "Not feature"),

    ("DEPARTURE_TIME", "Leakage", "Post-departure"),
    ("DEPARTURE_DELAY", "Leakage", "Post-departure"),
    ("TAXI_OUT", "Leakage", "Post-departure"),
    ("WHEELS_OFF", "Leakage", "Post-departure"),
    ("ELAPSED_TIME", "Leakage", "Post-departure"),
    ("AIR_TIME", "Leakage", "Post-departure"),
    ("WHEELS_ON", "Leakage", "Post-departure"),
    ("TAXI_IN", "Leakage", "Post-departure"),
    ("ARRIVAL_TIME", "Leakage", "Post-outcome"),
    ("CANCELLATION_REASON", "Leakage", "Post-outcome"),
    ("AIR_SYSTEM_DELAY", "Leakage", "Post-outcome"),
    ("SECURITY_DELAY", "Leakage", "Post-outcome"),
    ("AIRLINE_DELAY", "Leakage", "Post-outcome"),
    ("LATE_AIRCRAFT_DELAY", "Leakage", "Post-outcome"),
    ("WEATHER_DELAY", "Leakage", "Post-outcome"),
]

feature_roles_df = spark.createDataFrame(
    feature_roles,
    ["column", "role", "availability"]
)

feature_roles_df.show(40, truncate=False)

+-------------------+-----------------+-----------------------------+
|column             |role             |availability                 |
+-------------------+-----------------+-----------------------------+
|YEAR               |Candidate feature|Pre-departure                |
|MONTH              |Candidate feature|Pre-departure                |
|DAY                |Candidate feature|Pre-departure                |
|DAY_OF_WEEK        |Candidate feature|Pre-departure                |
|AIRLINE            |Candidate feature|Pre-departure                |
|FLIGHT_NUMBER      |Candidate feature|Pre-departure                |
|ORIGIN_AIRPORT     |Candidate feature|Pre-departure                |
|DESTINATION_AIRPORT|Candidate feature|Pre-departure                |
|SCHEDULED_DEPARTURE|Candidate feature|Pre-departure                |
|SCHEDULED_ARRIVAL  |Candidate feature|Pre-departure                |
|SCHEDULED_TIME     |Candidate feature|Pre-departure                |
|DISTANCE           

In [40]:
missing_decisions = [
    (
        "ARRIVAL_DELAY",
        "105,071 null rows",
        "Không có target cho regression/classification",
        "Dùng eligibility rule; không impute target",
        "Pipeline / Model",
    ),
    (
        "LATITUDE / LONGITUDE",
        "3 airport rows thiếu cả hai tọa độ",
        "Ảnh hưởng airport feature và Haversine",
        "Xác minh hoặc bổ sung ở module feature",
        "Custom/Feature",
    ),
    (
        "Delay-cause columns",
        "Khoảng 81.7% null",
        "Missing có ý nghĩa nghiệp vụ; đồng thời leakage",
        "Không tự fill 0; loại khỏi pre-departure feature set",
        "Custom/Feature",
    ),
    (
        "TAIL_NUMBER",
        "Có missing và cardinality cao",
        "Có thể ảnh hưởng encoding/feature stability",
        "Review availability và feature usefulness",
        "Custom/Feature",
    ),
    (
        "Actual-time columns",
        "Có missing theo trạng thái chuyến bay",
        "Không phù hợp inference trước departure",
        "Không dùng làm feature pre-departure",
        "Pipeline",
    ),
]

missing_decision_df = spark.createDataFrame(
    missing_decisions,
    [
        "column_or_group",
        "finding",
        "impact",
        "recommended_next_action",
        "owner",
    ],
)

missing_decision_df.show(truncate=False)

+--------------------+-------------------------------------+-----------------------------------------------+----------------------------------------------------+----------------+
|column_or_group     |finding                              |impact                                         |recommended_next_action                             |owner           |
+--------------------+-------------------------------------+-----------------------------------------------+----------------------------------------------------+----------------+
|ARRIVAL_DELAY       |105,071 null rows                    |Không có target cho regression/classification  |Dùng eligibility rule; không impute target          |Pipeline / Model|
|LATITUDE / LONGITUDE|3 airport rows thiếu cả hai tọa độ   |Ảnh hưởng airport feature và Haversine         |Xác minh hoặc bổ sung ở module feature              |Custom/Feature  |
|Delay-cause columns |Khoảng 81.7% null                    |Missing có ý nghĩa nghiệp vụ; đồng thời leaka

## 15. Data Quality Scorecard

Scorecard tổng hợp trạng thái chất lượng dữ liệu theo các dimension chính.

Không bắt buộc mọi dimension phải PASS.

In [41]:
scorecard = [
    (
        "Completeness",
        "PARTIAL",
        "Target, coordinates và một số post-outcome columns có missing",
        "Cần treatment theo từng nhóm ở module sau",
    ),
    (
        "Validity",
        "PASS",
        "Không phát hiện domain, HHMM hoặc coordinate-range violation",
        "Tiếp tục giữ validation trong pipeline",
    ),
    (
        "Uniqueness",
        "PARTIAL",
        "Lookup keys unique; full-row duplicate không chạy lại trong notebook",
        "Có thể chạy lại exact duplicate nếu cần audit đầy đủ",
    ),
    (
        "Consistency",
        "PASS",
        "Target missing phù hợp cancelled/diverted pattern",
        "Giữ business invariants khi xử lý sau",
    ),
    (
        "Referential Integrity",
        "PARTIAL",
        "Airline 100%; airport lookup khoảng 91.65%",
        "Cần verified DOT → IATA mapping",
    ),
    (
        "Accuracy",
        "UNASSESSED",
        "Không có external ground truth ở record level",
        "Không thể xác nhận accuracy tuyệt đối",
    ),
    (
        "Timeliness",
        "LIMITATION",
        "Dataset thuộc năm 2015",
        "Không đại diện trực tiếp cho hoạt động hàng không hiện tại",
    ),
]

scorecard_df = spark.createDataFrame(
    scorecard,
    [
        "dimension",
        "status",
        "evidence",
        "limitation_or_next_action",
    ],
)

scorecard_df.show(truncate=False)

+---------------------+----------+--------------------------------------------------------------------+----------------------------------------------------------+
|dimension            |status    |evidence                                                            |limitation_or_next_action                                 |
+---------------------+----------+--------------------------------------------------------------------+----------------------------------------------------------+
|Completeness         |PARTIAL   |Target, coordinates và một số post-outcome columns có missing       |Cần treatment theo từng nhóm ở module sau                 |
|Validity             |PASS      |Không phát hiện domain, HHMM hoặc coordinate-range violation        |Tiếp tục giữ validation trong pipeline                    |
|Uniqueness           |PARTIAL   |Lookup keys unique; full-row duplicate không chạy lại trong notebook|Có thể chạy lại exact duplicate nếu cần audit đầy đủ      |
|Consistency          

## 16. Final Conclusion và Handoff

### Kết luận chính

- Typed Parquet đã được đọc và profile thành công bằng PySpark.
- Schema và domain cơ bản hợp lệ.
- `ARRIVAL_DELAY` missing có pattern rõ ràng với cancelled/diverted flights.
- Classification target mất cân bằng với positive rate khoảng 11%.
- Airport lookup chỉ match khoảng 91.65% do xuất hiện mã DOT numeric ở tháng 10.
- Một số cột actual/post-outcome là data leakage nếu inference trước departure.
- Dataset hiện vẫn là **typed raw**, chưa phải cleaned hay model-ready.

### Handoff

**Custom/Feature**
- mapping DOT → IATA;
- xử lý 3 airport thiếu tọa độ;
- quyết định cuối cho `TAIL_NUMBER`;
- giữ danh sách leakage khi tạo feature.

**Pipeline**
- sử dụng typed schema;
- áp dụng eligibility contract;
- bảo đảm inference contract trước departure.

**Model**
- lưu ý regression target có giá trị âm;
- xử lý classification imbalance;
- dùng identical split khi so sánh model.

**Production**
- giữ schema input thống nhất;
- hỗ trợ hai dạng airport code nếu cần;
- streaming JSON sau này phải tương thích schema đã chốt.

In [42]:
spark.stop()

print("Spark session stopped.")

Spark session stopped.
